# Encoder Core ML -> head FP32 -> section labels
Mengukur dampak encoder aplikasi dibanding cache training dengan head FP32 yang
sama dan label manual yang sama. Ground truth tidak diubah. Audio memakai decoder/
resampler librosa pada Mac; ini belum menguji decoder Swift atau kecepatan iPhone.
Cache diagnostik terpisah dapat memakai beberapa GiB untuk 32 lagu. Tidak ada
training ulang. Mulai satu lagu, lalu seluruh validation, tanpa memakai test.


In [ ]:
import sys, subprocess, json
from pathlib import Path
from datetime import datetime
assert Path('finetuning/config.json').is_file(), 'Gunakan root Section-Annotations'
SONGMAP_ROOT = Path('/Users/danendra/Documents/ADA26/C6/SongMap')
proofs = sorted(Path('outputs.local').glob('coreml-comparison-*/fp32-validation.json'),
    key=lambda p:p.stat().st_mtime, reverse=True)
proofs = [p for p in proofs if (lambda r:r.get('full_validation') and r.get('logit_parity_passed')
    and r.get('precision_metadata') == 'fp32')(json.loads(p.read_text()))]
assert proofs, 'FP32 full validation yang lolos belum ditemukan'
proof = proofs[0]
report_dir = Path('outputs.local') / ('coreml-pipeline-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
report_dir.mkdir(exist_ok=False)
cache = Path('outputs.local/coreml-encoder-cache')
print('FP32 validation:',proof)
print('Diagnostic cache:',cache)
print('Report:',report_dir)

def run_logged(command, log_path):
    with log_path.open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if code:
        raise RuntimeError(f'Proses gagal ({code}); lihat {log_path}')

def evaluate_pipeline(limit, name):
    run_logged([sys.executable, '-u', '-m', 'deployment.evaluate_pipeline',
        '--songmap-root', str(SONGMAP_ROOT), '--fp32-validation-report', str(proof),
        '--cache-dir', str(cache), '--limit', str(limit),
        '--output', str(report_dir / f'{name}.json')], report_dir / f'{name}.log')


Satu lagu terlebih dahulu. Ini menjalankan encoder pada seluruh audio lagu, bukan hanya potongan30/120s. Window short/long kemudian digabung pada blok head420s seperti ekstraksi training.

In [ ]:
evaluate_pipeline(1,'one-track')
r = json.loads((report_dir/'one-track.json').read_text())
print('Reference F1:',r['reference_cache_with_fp32_head']['macro_f1_present'])
print('Core ML encoders F1:',r['coreml_encoders_with_fp32_head']['macro_f1_present'])
print('Delta:',r['macro_f1_delta'])
print('Section agreement:',r['section_prediction_agreement'])
print('Changed sections:',r['changed_sections'])


Jika satu lagu selesai tanpa runtime error, lanjut seluruh validation. Cache lagu pertama dilewati. Run ini dapat memerlukan waktu karena encoder dijalankan ulang sekali untuk fitur Core ML. Jika terputus, mulai setup baru lalu ulangi run; lagu dengan metadata lengkap akan digunakan kembali.

In [ ]:
evaluate_pipeline(0,'all-validation')
r = json.loads((report_dir/'all-validation.json').read_text())
print('Tracks:',r['tracks'],'| full validation:',r['full_validation'])
print('Reference F1:',r['reference_cache_with_fp32_head']['macro_f1_present'])
print('Core ML encoders F1:',r['coreml_encoders_with_fp32_head']['macro_f1_present'])
print('Delta:',r['macro_f1_delta'])
print('Section agreement:',r['section_prediction_agreement'])
print('Changed sections:',len(r['changed_sections']))
print('Per class:',r['coreml_encoders_with_fp32_head']['per_class'])
print('Report:',report_dir/'all-validation.json')
